In [1]:
from pyspark.sql import SparkSession
from delta import *

In [2]:
builder = SparkSession.builder.appName("Local_Lakehouse") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.sql.caseSensitive", "true")

In [3]:
spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/12 14:17:37 WARN Utils: Your hostname, LAPTOP-MSL0MUJB, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/12 14:17:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/harsh/wsl_venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/harsh/.ivy2.5.2/cache
The jars for the packages stored in: /home/harsh/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8638ee55-7e3c-4888-9dad-6f31ac9aa7e5;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache

In [4]:
df = spark.read \
    .format("json") \
    .load("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/getPosts/posts_results.jsonl")

In [5]:
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)

In [6]:
df.show(truncate=False,n=10)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
import requests
import time

In [8]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [9]:
# def test_send_request(uri):
#     # t1=time.time()
#     req=f"https://public.api.bsky.app/xrpc/app.bsky.feed.getPostThread?uri={uri}"
#     res = requests.get(req)
#     # t2=time.time()
#     # print(t2-t1)
#     # time.sleep(0.1)
#     #0.1 time skip not needed since t2-t1>1 sec
#     return res.json()


In [10]:
# send_request_udf=udf(test_send_request)

In [11]:
# df=df.withColumn("postThread",send_request_udf(col("uri")))

In [12]:
# df.show(n=10,truncate=False)

In [13]:
import os
import time
import json

In [14]:
RESULTS_FILE="/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/getPostThread/posts_thread_results.jsonl"
TRACKING_FILE="/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/getPostThread/processed_root_uris.txt"

processed_uris=set()

if(os.path.exists(TRACKING_FILE)):
    with open(TRACKING_FILE,"r") as f:
        processed_uris=set(line.strip() for line in f if line.strip())
try:
    unique_rows_df=df.select("uri").distinct().dropna()
    uris_prev=[row.uri for row in unique_rows_df.collect()]
    uris = [uri for uri in uris_prev if uri not in processed_uris]
except Exception as e:
    print(f"spark json schema error:{e}")

In [ ]:
num_retries=3
SLEEP_TIME=0.1
URL="https://public.api.bsky.app/xrpc/app.bsky.feed.getPostThread"
print(f"Found {len(uris)} new unique URIs to fetch.")
i=1
with open(TRACKING_FILE,"a") as tracking_file , open(RESULTS_FILE,"a") as results_file:
    for uri in uris:
        if i%5==0:
            print(f"uris done:{i}")
        if uri in processed_uris:
            continue
        for attempt in range(num_retries):
            try:
                # t1=time.time()
                res=requests.get(URL,params={"uri":uri})
                if res.status_code==429:
                    wait=int(res.headers.get("Retry-after",5))
                    time.sleep(wait)
                    continue
                elif res.status_code in (400, 404):
                    # We track the deleted URI as "processed" so we never try it again
                    tracking_file.write(f"{uri}\n")
                    tracking_file.flush()
                    processed_uris.add(uri)
                    break # Break out of retries instantly
                res.raise_for_status()
                if res.json() is not None:
                    results_file.write(json.dumps(res.json())+"\n")
                    results_file.flush()
                tracking_file.write(f"{uri}\n")
                tracking_file.flush()
                processed_uris.add(uri)
                # t2=time.time()
                # print(t2-t1)
                break
            except requests.exceptions.RequestException as e:
                print(f"error in attempt {attempt}:{e}")
                time.sleep(2)
        # time.sleep(SLEEP_TIME)
        i+=1

Found 2546748 new unique URIs to fetch.
uris done:5
uris done:10
uris done:15
uris done:20
uris done:25
uris done:30
uris done:35
uris done:40
uris done:45
uris done:50
uris done:55
uris done:60
uris done:65
uris done:70
uris done:75
uris done:80
uris done:85
uris done:90
uris done:95
uris done:100
uris done:105
uris done:110
uris done:115
uris done:120
uris done:125
uris done:130
uris done:135
uris done:140
uris done:145
uris done:150
uris done:155
uris done:160
uris done:165
uris done:170
uris done:175
uris done:180
uris done:185
uris done:190
uris done:195
uris done:200
uris done:205
uris done:210
uris done:215
uris done:220
uris done:225
uris done:230
uris done:235
uris done:240
uris done:245
uris done:250
uris done:255
uris done:260
uris done:265
uris done:270
uris done:275
uris done:280
uris done:285
uris done:290
uris done:295
uris done:300
uris done:305
uris done:310
uris done:315
uris done:320
uris done:325
uris done:330
uris done:335
uris done:340
uris done:345
uris done:350


26/04/12 16:20:06 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 4273109 ms exceeds timeout 120000 ms
26/04/12 16:20:06 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/12 16:20:07 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$